# COS60008 – Introduction to Data Science.
## Assignment – Australian Beach & Water Locations Dataset.

| | |
|---|---|
| **Student Name** | Bikram Bhattarai |
| **Student ID** | 105974087 |
| **Course** | Introduction to Data Sciencce |
| **Workshop Time** | Monday 12:30-02:30 PM |
| **Facilitator** | Nichole Ronald |
| **Semester** | Semester 1, 2026 |


## Setup – Imports and Configuration.

In [ ]:
# Standard libraries introduced in COS60008 — numpy, pandas, matplotlib for data manipulation and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Seed set for any operations requiring reproducibility.
np.random.seed(42)

# Global matplotlib style — no seaborn; spines and grid configured manually.
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
    'font.family': 'sans-serif'
})


---
## Task 1 – Data Acquisition & Preparation.


### 1.1 Load Raw Data Files.
Each source file is loaded into its own DataFrame. Shapes are then verified against
expected counts (raw line count minus the header row) to confirm no records were
dropped or duplicated during loading.

In [ ]:
# Each CSV loaded independently — no integration at this stage
df_nsw_raw       = pd.read_csv('data_nsw.csv')
df_other_raw     = pd.read_csv('data_except_nsw.csv')
df_amenities_raw = pd.read_csv('beach_amenities.csv')
df_climate       = pd.read_csv('climate_summary_by_region.csv')


In [ ]:
# Expected shapes derived from: wc -l <file> - 1 (header)
# data_nsw.csv:                  116 lines -> 115 data rows, 19 cols
# data_except_nsw.csv:           189 lines -> 188 data rows, 20 cols
# beach_amenities.csv:           304 lines -> 303 data rows,  6 cols
# climate_summary_by_region.csv:  70 lines ->  69 data rows,  7 cols
expected = {
    'df_nsw_raw':       (115, 19),
    'df_other_raw':     (188, 20),
    'df_amenities_raw': (303,  6),
    'df_climate':       ( 69,  7),
}

for name, df in [('df_nsw_raw', df_nsw_raw), ('df_other_raw', df_other_raw),
                  ('df_amenities_raw', df_amenities_raw), ('df_climate', df_climate)]:
    status = 'OK' if df.shape == expected[name] else 'MISMATCH'
    print(f'{name:<22}  expected {expected[name]}  loaded {df.shape}  [{status}]')


In [ ]:
# Column inspection — confirms NSW file is missing the State column (per data dictionary)
print('NSW columns:   ', df_nsw_raw.columns.tolist())
print('Other columns: ', df_other_raw.columns.tolist())


### 1.2 Data Integration
#### 1.2.1 Combining the Two Beach Files

In [ ]:
# State column is absent from the NSW file — added before concatenation
# so both DataFrames share an identical schema
df_nsw = df_nsw_raw.copy()
df_nsw.insert(4, 'State', 'NSW')


In [ ]:
# Four records carry non-Australian state codes (NZ / NF) and are removed
# as they fall outside the scope of an Australian beach dataset
non_au_mask = df_other_raw['State'].isin(['NZ', 'NF'])
print(f'Non-Australian records removed ({non_au_mask.sum()}):')
print(df_other_raw[non_au_mask][['Name', 'State', 'Region']].to_string(index=False))

df_other = df_other_raw[~non_au_mask].copy()

# Concatenate NSW and all-other-states into a single beach DataFrame
df_beaches = pd.concat([df_nsw, df_other], ignore_index=True)
print(f'\nCombined shape: {df_beaches.shape}')
print(f'States present: {sorted(df_beaches["State"].unique())}')


#### 1.2.2 Joining Amenities Data

In [ ]:
# Pre-join audit: check for name mismatches between beach and amenities files
df_amenities = df_amenities_raw.copy()
unmatched_before = set(df_beaches['Name']) - set(df_amenities['Name'])
print(f'Names in beach data with no amenities match: {unmatched_before}')


In [ ]:
# 'Noosa Main Bch' is an abbreviated form of 'Noosa Main Beach' — corrected before join
df_amenities['Name'] = df_amenities['Name'].replace({'Noosa Main Bch': 'Noosa Main Beach'})

unmatched_after = set(df_beaches['Name']) - set(df_amenities['Name'])
print(f'Unmatched names after correction: {len(unmatched_after)}')

# Left join retains all beach rows; amenity columns will be NaN where no match exists
df_main = df_beaches.merge(df_amenities, on='Name', how='left')
print(f'Shape after amenities join: {df_main.shape}')


#### 1.2.3 Joining Climate Data

In [ ]:
# Climate file uses 'Region_Name'; renamed to 'Region' to match the beach data join key
df_main = df_main.merge(
    df_climate.rename(columns={'Region_Name': 'Region'}),
    on='Region',
    how='left'   # left join — beach rows with no climate region should not be dropped
)
print(f'Shape after climate join: {df_main.shape}')


### 1.3 Data Cleaning

#### 1.3.1 Remove Duplicate Rows

In [ ]:
# Identify rows with the same beach Name — duplicates are checked for full-row equality
dup_names = df_main[df_main.duplicated('Name', keep=False)]['Name'].unique()
print('Duplicate beach names found:', dup_names)

# Both cases show identical coordinates and attribute values — confirmed true duplicates
print(df_main[df_main['Name'].isin(dup_names)][['Name','Latitude','Longitude','Annual_Visitors']])


In [ ]:
before = len(df_main)
# Keep only the first occurrence of each beach name
df_main = df_main.drop_duplicates(subset='Name', keep='first').reset_index(drop=True)
print(f'Rows removed: {before - len(df_main)} | Rows remaining: {len(df_main)}')


#### 1.3.2 Standardise Categorical Values

In [ ]:
# Full value_counts() audit on all object (categorical) columns
for col in df_main.select_dtypes('object').columns:
    vc = df_main[col].value_counts(dropna=False)
    if vc.index.astype(str).str.strip().nunique() < vc.nunique() or any(v in ['True','False','true','false'] for v in vc.index.astype(str)):
        print(f'--- {col} ---')
        print(vc.to_string())


In [ ]:
# Playground contains 'Yes ' (trailing space) and 'True' — both normalised to 'Yes'
df_main['Playground'] = df_main['Playground'].str.strip().replace({'True': 'Yes'})
print('Playground after standardisation:')
print(df_main['Playground'].value_counts(dropna=False))


#### 1.3.3 Missing Value Audit and Imputation

In [ ]:
# Audit — count and percentage of missing values per column
missing     = df_main.isnull().sum()
pct_missing = (missing / len(df_main) * 100).round(1)
audit = pd.DataFrame({'missing_count': missing, 'pct_missing': pct_missing})
audit = audit[audit['missing_count'] > 0].sort_values('pct_missing', ascending=False)
print(audit.to_string())


In [ ]:
# Dogs_Allowed (59.9%) and Flags (54.9%) both exceed the 50% drop threshold
df_main = df_main.drop(columns=['Dogs_Allowed', 'Flags'])
print('Columns dropped: Dogs_Allowed, Flags')


In [ ]:
# Has_Showers (3.0%) — binary field; imputed with the mode value
showers_mode = df_main['Has_Showers'].mode()[0]
df_main['Has_Showers'] = df_main['Has_Showers'].fillna(showers_mode)
print(f'Has_Showers: imputed with mode={showers_mode!r}, remaining missing: {df_main["Has_Showers"].isnull().sum()}')


In [ ]:
# Beach_Length_km (3.0%) — length varies significantly by beach type,
# so a grouped median (per Beach_Type) is more appropriate than the global median
df_main['Beach_Length_km'] = (
    df_main.groupby('Beach_Type')['Beach_Length_km']
    .transform(lambda x: x.fillna(x.median()))
)
print(f'Beach_Length_km missing after impute: {df_main["Beach_Length_km"].isnull().sum()}')


In [ ]:
# Water_Quality_Rating and General_Hazard_Rating (2.0% each)
# Both are ordinal 1-5 integer scales; global median preserves the rating distribution
wq_med = df_main['Water_Quality_Rating'].median()
hz_med = df_main['General_Hazard_Rating'].median()

df_main['Water_Quality_Rating']  = df_main['Water_Quality_Rating'].fillna(wq_med).astype(int)
df_main['General_Hazard_Rating'] = df_main['General_Hazard_Rating'].fillna(hz_med).astype(int)

print(f'Water_Quality_Rating  filled (median={wq_med}), missing: {df_main["Water_Quality_Rating"].isnull().sum()}')
print(f'General_Hazard_Rating filled (median={hz_med}), missing: {df_main["General_Hazard_Rating"].isnull().sum()}')


In [ ]:
# Final verification — confirm zero missing values remain
total_missing = df_main.isnull().sum().sum()
print(f'Total missing values remaining: {total_missing}')
print(f'Final df_main shape: {df_main.shape}')
print(f'Columns: {df_main.columns.tolist()}')


### 1.4 Final DataFrame Preview

In [ ]:
df_main.head()

In [ ]:
df_main.dtypes

---
## Task 2 – Data Exploration
*To be completed in the next stage.*
